4.3 Use Random Forest to determine which features are essential for the machine learning model and categorical data in my_data

Create training and validation sets

In [ ]:
X_train = my_data[(my_data['Date'] < '2013-09-01') & (my_data['Date'] > '2013-01-01') ]
y_train = X_train['Sales']
X_train = X_train.drop('Sales' , axis = 1 )
X_valid = my_data[(my_data['Date'] >='2013-09-01') & (my_data['Date'] < '2014-01-01') ]
y_valid = X_valid['Sales']
X_valid = X_valid.drop('Sales' , axis = 1 )

Setup pipeline

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import category_encoders as ce
numrical_cols = X_train.select_dtypes(exclude = 'object').columns
categorical_cols = X_train.select_dtypes(include = 'object').columns
numrical = Pipeline(steps = [('impute' , SimpleImputer(strategy = 'constant' , fill_value = 0))])
categorical = Pipeline(steps = [('impute' , SimpleImputer(strategy = 'constant' , fill_value = 'missing')) ,
                                ('Target_encoders' , ce.TargetEncoder(smoothing = 10.0))])
preprocessing = ColumnTransformer( transformers = [('num' , numrical , numrical_cols) ,
                                                 ('cate' , categorical , categorical_cols)])
print('Create pipeline complete!')

Create Random Forest model and link model with pipeline

In [ ]:
from sklearn.ensemble import RandomForestRegressor
X_train[categorical_cols] = X_train[categorical_cols].astype(str)
model_test = RandomForestRegressor( n_estimators = 100 , random_state = 0 , n_jobs=-1, max_depth=20 , verbose = False)
model_pipeline = Pipeline ( steps = [('preprocessing' , preprocessing) ,
                                     ('model' , model_test)])
model_pipeline = model_pipeline.fit(X_train , y_train)
print('complete train model')

Evaluate which features are truly useful

In [ ]:
result_step = model_pipeline.named_steps['model']
score = result_step.feature_importances_
cat_features_new =  model_pipeline.named_steps['preprocessing'].named_transformers_['cate'].named_steps['Target_encoders'].get_feature_names_out(categorical_cols)
all_feature_names = list(numrical_cols) + list(cat_features_new)
result_df = pd.DataFrame({'Features': all_feature_names, 'Score': score}).sort_values(by='Score', ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x='Score', y='Features', data=result_df.head(15))
plt.title('The impact of top 15 features on sales')
plt.tight_layout()
plt.show()

Console: The columns with low correlation PromoInterval and Promo2SinceYear need to be removed. Regarding the Assortment columns: although the correlations forAssortment a and Assortment c are low, they belong to the same feature as Assortment b (which shows high correlation), so the Assortment column should be retained.

**5. Mutual Information**

Set up the data required to find mutual information.

In [ ]:
X = my_data[(my_data['Date'] < '2013-09-01') & (my_data['Date'] > '2013-01-01') ]
y = X['Sales']
X = X.drop(['Sales', 'Date'] , axis = 1 )
for colname in X.select_dtypes('object'):
    X[colname] , _ = X[colname].factorize()
discrete_features = X.dtypes == int
print('complete!')

check data reciprocity

In [ ]:
from sklearn.feature_selection import mutual_info_regression
mi_scores = mutual_info_regression(X , y , discrete_features = discrete_features)
mi_scores = pd.Series( mi_scores , name = 'MI Scores' , index = X.columns)
mi_scores = mi_scores.sort_values( ascending = False)
mi_scores[::3]

Visualize MI scores on a bar plot

In [ ]:
plt.figure( dpi=100 , figsize = ( 8 , 5 ))
scores = mi_scores.sort_values(ascending = True )
width = np.arange(len(scores))
ticks = list(scores.index)
plt.barh(width , scores)
plt.yticks(width , ticks)
plt.title('Mutual Information Scores')
plt.tight_layout()

Console: Here, we observe a column representing customer volume with unusually high values,since customer numbers are directly proportional to revenue, we should remove the customer column but retain the remaining ones. Even if their individual correlation scores are low, combining them with other variables enhances their value within the predictive model

**6.Feature Selection**

We begin by removing noisy and unnecessary features (such as identifiers).

In [ ]:
my_data = my_data.drop(['PromoInterval' , 'Customers'] , axis = 1 )
print(my_data.head())

# **Feature Enginnering**

**1. create new features useful for predict model**

We start by splitting the Date column so the model can better understand the date information. Later, we need to remove Date columns because it is no longer useful

In [ ]:
my_data['Date'] = pd.to_datetime( my_data['Date'])
my_data['Day'] = my_data['Date'].dt.day
my_data['Month'] = my_data['Date'].dt.month
my_data['Year'] = my_data['Date'].dt.year
my_data = my_data.drop('Date' , axis = 1 )
print(my_data.head())

Create an additional feature regarding competitors' store operating hours so the model can identify revenue trends in the presence of competing stores. Later , we proceed remove CompetitionOpenSinceMonth , CompetitionOpenSinceYear because it is not longer useful

In [ ]:
my_data['CompetitorLongevity'] = my_data['Year'] - my_data['CompetitionOpenSinceYear']
my_data.loc[my_data['CompetitionOpenSinceYear'] == 0, 'CompetitorLongevity'] = 0
my_data = my_data.drop([ 'CompetitionOpenSinceMonth' , 'CompetitionOpenSinceYear'] , axis = 1)
print(my_data.head())

**2. Time Series processing**

2.1 Trend feature

smooth the data and observe long-term trends

In [ ]:
my_data['Date'] = pd.to_datetime(my_data[['Year', 'Month', 'Day']])
daily_sales = my_data.groupby('Date')['Sales'].mean()
moving_average = daily_sales.rolling(
    window = 365 ,
    center = True ,
    min_periods = 183
).mean()
ax = daily_sales.plot(style=".", color="0.5")
moving_average.plot(
    ax=ax, linewidth=3, title="Rossmann store sales - 365-Day Moving Average", legend=False,
);

The moving average does not move sideways or trend downward but instead rises steadily over time remaining largely unaffected by noise indicating that the chain of stores is on a growth trajectory. Meanwhile, the uniform dispersion of data suggests a highly consistent business operational pattern across the years.

engineer additional trend-related data features

In [ ]:
from statsmodels.tsa.deterministic import DeterministicProcess
daily_sales.index = daily_sales.index.to_period('D')
dp = DeterministicProcess(
    index = daily_sales.index ,
    constant = True ,
    order = 1 ,
    drop = True
)
my_data['Date'] = my_data['Date'].dt.to_period('D')
X_trend = dp.in_sample()
print(X_trend.head())

visualize long-term trends

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression(fit_intercept=False)
y = daily_sales 
model.fit(X_trend,y)
y_pred = pd.Series(model.predict(X_trend), index=X_trend.index)
plt.figure(figsize=(10, 5))
ax = y.plot(style=".", color="0.5", alpha=0.5, label="Actual Sales")
y_pred.plot(ax=ax, linewidth=3, color="blue", label="Linear trend")
plt.title("Rossmann Store Sales - Evaluation Linear Regression model")
plt.legend()
plt.show()

Try forecasting revenue for the next 30 days

In [ ]:
X_trend = dp.out_of_sample(steps=30)
y_fore = pd.Series(model.predict(X_trend), index=X_trend.index)
plt.figure(figsize=(12, 5))
ax = y.plot(style=".-", color="0.5", alpha=0.6, label="Sales")
y_pred.plot(ax=ax, linewidth=3, color="blue", label="Trend")
y_fore.plot(ax=ax, linewidth=3, color="red", label="Trend Forecast")
plt.title("Daily sales - Linear Trend Forecast")
plt.legend()
plt.show()

2.2 Seasonality

Plot a chart showing the impact of dates on revenue.

In [ ]:
def seasonal_plot( X , y , period , freq , ax=None ):
    if ax is None:
        fig, ax = plt.subplots()
    sns.lineplot( data = X , x = freq , y = y , hue = period , errorbar = None , palette="husl" , ax = ax , legend = False)
    ax.set_title(f'Seasonal Plot ({period}/{freq})')
    return ax
X_season = daily_sales.to_frame(name='Sales')
X_season['day'] = X_season.index.dayofweek
X_season['week'] = X_season.index.week
X_season['dayofyear'] = X_season.index.dayofyear
X_season['year'] = X_season.index.year
fig, (ax0 , ax1) = plt.subplots(2 , 1 , figsize = (11 , 6))
seasonal_plot(X_season , y = 'Sales' , period = 'week' , freq = 'day' , ax = ax0 )
seasonal_plot(X_season , y = 'Sales' , period = 'year' , freq = 'dayofyear' , ax = ax1 )
plt.tight_layout()

The chart clearly illustrates the cyclical nature of the store chain's revenue: sales peak on Sundays and drop sharply on Saturdays, while remaining relatively stable throughout the rest of the week

Now let's look at the periodogram:

In [ ]:
from scipy.signal import periodogram
def plot_periodogram(ts, detrend='linear', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    frequencies, variance = periodogram( ts, detrend=detrend, scaling='spectrum')
    ax.step(frequencies, variance, color="purple")
    ax.set_title('periodogram - Rossmann Store Sales')
    periods = [365, 30, 7, 3.5]
    x_ticks = [1 / p for p in periods]
    x_labels = ['Annual (1)', 'Monthly (12)', 'Weekly (52)', 'Semiweekly (104)']
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=30)
    return ax
plot_periodogram( X_season['Sales'] )

create seasonality feature 

In [ ]:
from statsmodels.tsa.deterministic import CalendarFourier
fourier_annual = CalendarFourier(freq = 'YE' , order = 10 )
fourier_month = CalendarFourier(freq = 'W' , order = 2 )
dp = DeterministicProcess(
    index = X_season.index ,
    constant = True ,
    order = 1,
    seasonal = True ,
    additional_terms = [ fourier_annual , fourier_month ] ,
    drop = True
)
X_fourier = dp.in_sample()
print(X_fourier.head())

Visualize seasonality and attempt to forecast sales for the next 30 days

In [ ]:
from sklearn.linear_model import Lasso
model_lasso = Lasso(alpha = 1.0)
model.fit(X_fourier , y)
y_pred = pd.Series( model.predict(X_fourier) , index = X_fourier.index)
X_for = dp.out_of_sample(steps = 30)
y_fore = pd.Series( model.predict(X_for) , index = X_for.index)
plt.figure(figsize=(12, 5))
ax = y.plot(style=".-", color="0.5", alpha=0.7, label="Sales")
y_pred.plot(ax=ax, linewidth=1, color="blue", label="Season")
y_fore.plot(ax=ax, linewidth=1, color="red", label="Season Forecast")
plt.title("Daily sales - Linear Season Forecast")
plt.legend()

Add trend and fourier feature to the data table

In [ ]:
my_data = pd.merge( my_data , X_fourier , how = 'left' , on = 'Date' )
print(my_data.head())

4.3 Create time series feature

Revenue adjusted for trend and seasonal factors

In [ ]:
y_deseason = y - y_pred 
y_deseason.name = 'Sale Deseasoned'
ax = y_deseason.plot()
ax.set_title('Sales of Rossman Store Sales(Deseasoned)')

When unaffected by seasonality, the data is driven solely by random noise or promotions.

perform weekly data smoothing with y

In [ ]:
y_ma = y.rolling( window = 7 , center = True ).mean()
ax = y_ma.plot()
ax.set_title('Seven day moving average')

The moving average plot exhibits significant amplitude, clearly reflecting annual revenue surge cycles, whereas the residual plot shows reduced amplitude; thus, we conclude that the model captures over 80% of the data variation, with the remaining residuals representing only outliers and noise

Read the remaining rules using lag

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf
fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(y_deseason, lags=8, ax=ax);

In [ ]:
def my_plot_lags ( y , lags = 8 , nrows = 2 ):
    ncols = lags//nrows
    fig , axes = plt.subplots( nrows = nrows , ncols = ncols , figsize = (16 , 6) , sharex = True , sharey = True )
    axes = axes.flatten()
    for i in range(1 , lags + 1):
        ax = axes[i-1]
        y_lag = y.shift(i)
        sns.regplot( x = y_lag, y = y , ax = ax , scatter_kws = {'s': 5 , 'alpha': 0.7 } ,line_kws={'color': 'red'}, lowess=True )
        ax.set_title ( f'Lag {i}')
        ax.set_xlabel ('Sales deseasoned')
        if i % ncols == 1:
            ax.set_ylabel('sales_deseasoned')
        else:
            ax.set_ylabel('')
    plt.tight_layout()
my_plot_lags( y_deseason , lags = 8 , nrows = 2)

The Partial Autocorrelation plot shows that most lags exceed the blue threshold. however, this is due to the narrow error margin rather than the lags actually being useful. Upon closer inspection of the individual lags, most appear flat or fluctuate in response to noise, whereas only lag 1 and lag 8 prove useful for prediction being relatively unaffected by noise so we proceed to add these lags to my_data.

In [ ]:
my_lags = pd.DataFrame(index=y_deseason.index)
my_lags['lag_1_resid'] = y_deseason.shift(1).values
my_lags['lag_8_resid'] = y_deseason.shift(8).values
my_data['lag_1'] = my_data.groupby('Store')['Sales'].shift(1)
my_data['lag_8'] = my_data.groupby('Store')['Sales'].shift(8)
my_lags = my_lags.reset_index()
if 'index' in my_lags.columns:
    my_lags = my_lags.rename(columns={'index': 'Date'})
my_data = pd.merge(my_data, my_lags, how='left', on='Date')
my_data = my_data.dropna()
print(my_data[['Date', 'Store', 'lag_1', 'lag_8' , 'lag_1_resid' , 'lag_8_resid']].head())

**3. Process data to train the model**

3.1 preparing data for the hybrid model

After deriving the patterns regarding seasonal trends, the 'Date' column is no longer necessary, so we remove it; at the same time, we delete any empty columns that introduce noise into the data.

In [ ]:
my_data = my_data.drop('Date' , axis = 1 )
my_data = my_data.dropna()
print(my_data.head())

Create Boosted Hybrid class and related functions.

In [ ]:
from sklearn.base import BaseEstimator
class BoostedHybrid(BaseEstimator):
    def __init__( self , model_1 , model_2 ):
        self.model_1 = model_1 
        self.model_2 = model_2
        self.y_columns = None
def fit( self , X_1 , X_2 , y ):
    self.model_1.fit(X_1 , y)
    y_pred = self.model_1.predict(X_1)
    y_fit = pd.DataFrame( y_pred , index = X_1.index , columns = y.columns )
    y_resid = y - y_fit
    y_resid = y_resid.stack().squeeze()
    self.model_2.fit(X_2 , y_resid)
    self.y_columns = y.columns
    self.y_fit = y_fit
    self.y_resid = y_resid
    return self
BoostedHybrid.fit = fit
def predict( self , X_1 , X_2 ):
    y_pred = pd.DataFrame( self.model_1.predict(X_1) , index = X_1.index , columns = self.y_columns )
    y_pred = y_pred.stack().squeeze()
    y_pred += self.model_2.predict(X_2)
    return y_pred.unstack()
BoostedHybrid.predict = predict
print('BoostHybrid class was completed!')

Create a dataset for training machine learning models.

In [ ]:
feature_hd = ['Store' , 'StoreType' , 'Assortment' , 'CompetitionDistance' , 'CompetitorLongevity' , 'Promo2', 'Promo2SinceWeek' , 'Promo2SinceYear' , 'DayOfWeek' , 'Promo' , 'StateHoliday' , 'SchoolHoliday' , 'Day' , 'Month' , 'lag_1_resid' , 'lag_8_resid' , 'Year']
mask_train = (my_data['Year'] < 2015)
X_train_hd = my_data[mask_train]
X_valid_hd = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 1) & (my_data['Month'] < 5)]
X_test_hd = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 5)]
y_train_hd = X_train_hd[['Sales']]
X_train_hd = X_train_hd.drop('Sales' , axis = 1 )
y_valid_hd = X_valid_hd[['Sales']]
X_valid_hd = X_valid_hd.drop('Sales' , axis = 1 )
y_test_hd = X_test_hd[['Sales']]
X_test_hd = X_test_hd.drop('Sales' , axis = 1 )
X_test_1 = X_test_hd.drop( feature_hd , axis = 1 )
X_test_2 = X_test_hd[feature_hd]
X_train_1 = X_train_hd.drop( feature_hd, axis = 1 )
X_train_2 = X_train_hd[feature_hd]
X_valid_1 = X_valid_hd.drop( feature_hd , axis = 1 )
X_valid_2 = X_valid_hd[feature_hd]
print('setup data set complete!')

Handle categorical features using target_encoder.

In [ ]:
target_encoder = ce.TargetEncoder(smoothing = 10.0)
X_train_2 = target_encoder.fit_transform(X_train_2 , y_train_hd)
X_valid_2 = target_encoder.transform(X_valid_2)
X_test_2 = target_encoder.transform(X_test_2)
print('Complete!')

3.2 preparing data for a single LGBM model

Create train model data

In [ ]:
mask_train = (my_data['Year'] < 2015) | ((my_data['Year'] == 2015) & (my_data['Month'] < 2))
X_train = my_data[mask_train]
X_valid = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 1) & (my_data['Month'] < 5)]
X_test = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 5) ]
y_train = X_train['Sales']
X_train = X_train.drop('Sales' , axis = 1 )
y_valid = X_valid['Sales']
X_valid = X_valid.drop('Sales' , axis = 1 )
y_test = X_test['Sales']
X_test = X_test.drop('Sales' , axis = 1 )
feature_lgb = ['Store' , 'StoreType' , 'Assortment' , 'CompetitionDistance' , 'CompetitorLongevity' , 'Promo2', 'Promo2SinceWeek' , 'Promo2SinceYear' , 'DayOfWeek' , 'Promo' , 'StateHoliday' , 'SchoolHoliday' , 'Day' , 'Month' , 'lag_1' , 'lag_8' , 'Year']
X_train = X_train[feature_lgb]
X_valid = X_valid[feature_lgb]
X_test = X_test[feature_lgb]
print('setup data set complete!')

processing data

In [ ]:
X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
X_test = target_encoder.transform(X_test)
print('complete!')

**4. Data clustering**

Select features for clustering

In [ ]:
features_1 = ['CompetitionDistance' , 'CompetitorLongevity']
X_train_scale_1 = X_train.loc[: , features_1]
X_valid_scale_1 = X_valid.loc[: , features_1]
X_test_scale_1 = X_test.loc[: , features_1]
X_train_2_scale_1 = X_train_2.loc[: , features_1]
X_valid_2_scale_1 = X_valid_2.loc[: , features_1]
X_test_2_scale_1 = X_test_2.loc[: , features_1]
features_2 = ['StoreType' , 'Assortment']
X_train_scale_2 = X_train.loc[: , features_2]
X_valid_scale_2 = X_valid.loc[: , features_2]
X_test_scale_2 = X_test.loc[: , features_2]
X_train_2_scale_2 = X_train_2.loc[: , features_2]
X_valid_2_scale_2 = X_valid_2.loc[: , features_2]
X_test_2_scale_2 = X_test_2.loc[: , features_2]
print('complete!')

normalize parameters for clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
scale_1 = StandardScaler().set_output(transform="pandas")
scale_2 = StandardScaler().set_output(transform="pandas")
scale_3 = StandardScaler().set_output(transform="pandas")
scale_4 = StandardScaler().set_output(transform="pandas")
X_train_scale_1 = scale_1.fit_transform(X_train_scale_1)
X_valid_scale_1 = scale_1.transform(X_valid_scale_1)
X_test_scale_1 = scale_1.transform(X_test_scale_1)
X_train_2_scale_1 = scale_3.fit_transform(X_train_2_scale_1)
X_valid_2_scale_1 = scale_3.transform(X_valid_2_scale_1)
X_test_2_scale_1 = scale_3.transform(X_test_2_scale_1)
X_train_scale_2 = scale_2.fit_transform(X_train_scale_2)
X_valid_scale_2 = scale_2.transform(X_valid_scale_2)
X_test_scale_2 = scale_2.transform(X_test_scale_2)
X_train_2_scale_2 = scale_4.fit_transform(X_train_2_scale_2)
X_valid_2_scale_2 = scale_4.transform(X_valid_2_scale_2)
X_test_2_scale_2 = scale_4.transform(X_test_2_scale_2)
print('complete!')

Using K mean algorithm to clusters features

In [ ]:
from sklearn.cluster import KMeans
kmean_1 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_2 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_3 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_4 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
X_train['Cluster_1'] = kmean_1.fit_predict(X_train_scale_1)
X_valid['Cluster_1'] = kmean_1.predict(X_valid_scale_1)
X_test['Cluster_1'] = kmean_1.predict(X_test_scale_1)
X_train['Cluster_1'] = X_train['Cluster_1'].astype('string')
X_valid['Cluster_1'] = X_valid['Cluster_1'].astype('string')
X_test['Cluster_1'] = X_test['Cluster_1'].astype('string')
X_train['Cluster_2'] = kmean_2.fit_predict(X_train_scale_2)
X_valid['Cluster_2'] = kmean_2.predict(X_valid_scale_2)
X_test['Cluster_2'] = kmean_2.predict(X_test_scale_2)
X_train['Cluster_2'] = X_train['Cluster_2'].astype('string')
X_valid['Cluster_2'] = X_valid['Cluster_2'].astype('string')
X_test['Cluster_2'] = X_test['Cluster_2'].astype('string')
X_train_2['Cluster_1'] = kmean_3.fit_predict(X_train_2_scale_1)
X_valid_2['Cluster_1'] = kmean_3.predict(X_valid_2_scale_1)
X_test_2['Cluster_1'] = kmean_3.predict(X_test_2_scale_1)
X_train_2['Cluster_1'] = X_train_2['Cluster_1'].astype('string')
X_valid_2['Cluster_1'] = X_valid_2['Cluster_1'].astype('string')
X_test_2['Cluster_1'] = X_test_2['Cluster_1'].astype('string')
X_train_2['Cluster_2'] = kmean_4.fit_predict(X_train_2_scale_2)
X_valid_2['Cluster_2'] = kmean_4.predict(X_valid_2_scale_2)
X_test_2['Cluster_2'] = kmean_4.predict(X_test_2_scale_2)
X_train_2['Cluster_2'] = X_train_2['Cluster_2'].astype('string')
X_valid_2['Cluster_2'] = X_valid_2['Cluster_2'].astype('string')
X_test_2['Cluster_2'] = X_test_2['Cluster_2'].astype('string')
cluster_encoder_1 = ce.TargetEncoder(smoothing = 10.0)
cluster_encoder_2 = ce.TargetEncoder(smoothing = 10.0)
cluster_cols = ['Cluster_1' , 'Cluster_2']
X_train[cluster_cols] = cluster_encoder_1.fit_transform(X_train[cluster_cols] , y_train)
X_valid[cluster_cols] = cluster_encoder_1.transform(X_valid[cluster_cols])
X_test[cluster_cols] = cluster_encoder_1.transform(X_test[cluster_cols])
X_train_2[cluster_cols] = cluster_encoder_2.fit_transform( X_train_2[cluster_cols] , y_train_hd)
X_valid_2[cluster_cols] = cluster_encoder_2.transform(X_valid_2[cluster_cols])
X_test_2[cluster_cols] = cluster_encoder_2.transform(X_test_2[cluster_cols])
print('complete!')